# Lab4.3 — Resize în software

Redimensionarea imaginilor este o operație frecventă înaintea procesării sau inferenței AI. În acest task analizăm costul ei pe ARM.


## Obiective

- redimensionarea unei imagini cu OpenCV;
- compararea metodelor de interpolare;
- măsurarea timpului de execuție;
- observarea efectului rezoluției asupra volumului de date.


In [ ]:
from pathlib import Path

IMAGE_PATH = Path("images/fruits.jpg")

if not IMAGE_PATH.exists():
    raise FileNotFoundError(
        "Nu a fost găsită imaginea Lab4/images/fruits.jpg. "
        "Verificați că repository-ul a fost copiat complet pe placă."
    )

print("Image:", IMAGE_PATH)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import time

image_bgr = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_COLOR)
if image_bgr is None:
    raise RuntimeError("Imaginea nu a putut fi încărcată.")

## 1. Redimensionare la 320 × 240

În OpenCV, dimensiunea se specifică în ordinea `(width, height)`.


In [ ]:
target_size = (320, 240)
resized = cv2.resize(image_bgr, target_size, interpolation=cv2.INTER_LINEAR)

print("Original:", image_bgr.shape)
print("Resized :", resized.shape)
print("Original bytes:", image_bgr.nbytes)
print("Resized bytes :", resized.nbytes)

In [ ]:
plt.figure(figsize=(7, 4))
plt.imshow(cv2.cvtColor(resized, cv2.COLOR_BGR2RGB))
plt.title("Resize 320 × 240")
plt.axis("off")
plt.show()

## 2. Interpolare

Comparăm trei metode:

- `INTER_NEAREST` — simplă și rapidă;
- `INTER_LINEAR` — compromis uzual;
- `INTER_CUBIC` — mai costisitoare.


In [ ]:
methods = {
    "NEAREST": cv2.INTER_NEAREST,
    "LINEAR": cv2.INTER_LINEAR,
    "CUBIC": cv2.INTER_CUBIC,
}

REPEATS = 50

for name, method in methods.items():
    cv2.resize(image_bgr, target_size, interpolation=method)
    t0 = time.perf_counter()
    for _ in range(REPEATS):
        tmp = cv2.resize(image_bgr, target_size, interpolation=method)
    mean_t = (time.perf_counter() - t0) / REPEATS
    print(f"{name:8s}: {mean_t*1e3:8.3f} ms  -> {1/mean_t:8.1f} FPS")

## 3. Costul pentru mai multe rezoluții de ieșire


In [ ]:
targets = ((160, 120), (320, 240), (640, 480), (1280, 720))

for w, h in targets:
    t0 = time.perf_counter()
    for _ in range(REPEATS):
        tmp = cv2.resize(image_bgr, (w, h), interpolation=cv2.INTER_LINEAR)
    mean_t = (time.perf_counter() - t0) / REPEATS
    print(f"{w:4d}x{h:<4d}: {mean_t*1e3:8.3f} ms  {1/mean_t:8.1f} FPS")

## 4. Exercițiu

Redimensionați imaginea la **640 × 360** și determinați:

- numărul de pixeli;
- memoria ocupată;
- timpul mediu pentru 100 de redimensionări;
- FPS teoretic.


In [ ]:
# Scrieți aici soluția


## 5. Întrebări

1. De ce redimensionarea la o rezoluție mai mare necesită în general mai mult calcul?
2. Ce compromis există între metoda de interpolare și costul de calcul?
3. De ce resize este un candidat bun pentru accelerare hardware într-un pipeline video?
